In [8]:
import pandas as pd, numpy as np, torch

In [11]:
from transformers import AutoTokenizer, AutoModel

In [6]:
manifest = pd.read_csv('../data/manifest.csv')

In [24]:
MODEL = 'roberta-base'
BATCH = 64
dev = "mps" if torch.backends.mps.is_available() else "cpu"1

tok = AutoTokenizer.from_pretrained(MODEL)
enc = AutoModel.from_pretrained(MODEL).to(dev)
enc.eval()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [25]:
manifest.Utterance[0]

'also I was the point person on my company’s transition from the KL-5 to GR-6 system.'

In [45]:
texts = manifest.Utterance.fillna("").astype(str).tolist()
output = []


batch_size = 64
with torch.no_grad():    
    for i in range(0, len(texts), batch_size):
        b = tok(texts[i:i+batch_size], padding=True,truncation=True, max_length=128,return_tensors="pt")
        b = b.to(dev)
        h = enc(**b).last_hidden_state 
        m = b["attention_mask"].unsqueeze(-1).float()
        output.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())   # pool all vectors into 1 (one sentance could have 6 vectors another could have 20)
        print(f"{min(i+BATCH, len(texts))}/{len(texts)}", end="\r")

X = np.concatenate(output)
print(X.shape)   # (13706, 768)
        #outputs = enc(**batch)   # or model(**batch).last_hidden_state

(13706, 768)


In [47]:
X[0]

array([ 6.96187885e-03,  7.18907937e-02, -2.27901936e-02, -5.27513102e-02,
       -6.70753568e-02,  9.64385718e-02, -2.91175265e-02,  5.62360287e-02,
       -1.78902429e-02,  1.00150304e-02,  2.96265613e-02, -3.81728485e-02,
        9.28991511e-02,  1.01945929e-01,  7.42021874e-02, -6.33528233e-02,
        8.17802474e-02, -8.72713700e-03, -9.76777151e-02,  2.40084436e-02,
        1.11976869e-01, -3.61399651e-02, -8.66344497e-02, -2.58933790e-02,
       -4.70489413e-02,  1.10002505e-02, -4.95801447e-03,  5.57961464e-02,
       -4.48723265e-04, -1.24621086e-01, -1.22886293e-01,  3.18077058e-02,
       -5.56015894e-02,  6.09914921e-02, -6.13532104e-02,  6.67600632e-02,
        1.39870137e-01,  1.90742686e-02, -2.00966597e-02, -3.39265019e-02,
        1.69582993e-01, -1.20387994e-01, -2.24615615e-02, -5.43728881e-02,
       -2.23525632e-02,  8.87163356e-02, -4.36034389e-02, -8.98013487e-02,
        1.07896678e-01, -9.99760926e-02, -2.23978870e-02,  2.97824219e-02,
       -3.73473763e-03,  

In [36]:
manifest.Utterance.tolist()

['also I was the point person on my company’s transition from the KL-5 to GR-6 system.',
 'You must’ve had your hands full.',
 'That I did. That I did.',
 'So let’s talk a little bit about your duties.',
 'My duties?  All right.',
 'Now you’ll be heading a whole division, so you’ll have a lot of duties.',
 'I see.',
 'But there’ll be perhaps 30 people under you so you can dump a certain amount on them.',
 'Good to know.',
 'We can go into detail',
 'No don’t I beg of you!',
 'All right then, we’ll have a definite answer for you on Monday, but I think I can say with some confidence, you’ll fit in well here.',
 'Really?!',
 'Absolutely.  You can relax',
 'But then who? The waitress I went out with last month?',
 'You know? Forget it!',
 'No-no-no-no, no! Who, who were you talking about?',
 "No, I-I-I-I don't, I actually don't know",
 'Ok!',
 'All right, well...',
 'Yeah, sure!',
 'Hey, Mon.',
 'Hey-hey-hey. You wanna hear something that sucks.',
 'Do I ever.',
 'Chris says they’re closin